### Text To Sql Agent

In [9]:
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
import os 
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [11]:
DB_USER=os.getenv("DB_USER")
DB_NAME=os.getenv("DB_NAME")
DB_PASSWORD=os.getenv("DB_PASSWORD")
DB_PORT=os.getenv("DB_PORT")
DB_HOST=os.getenv("DB_HOST")

In [12]:
from sqlalchemy.engine import URL

connection_url=URL.create(
    "mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

In [13]:
engine=create_engine(
    connection_url
)

In [15]:
db=SQLDatabase(
    engine,
    sample_rows_in_table_info=0,
    ignore_tables=["_demodb_import"]
)

In [32]:
def get_schema():
    return db.get_table_info()

In [33]:
schema=schema()

TypeError: 'str' object is not callable

In [34]:
print(schema)


CREATE TABLE album (
	`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Title` VARCHAR(160) NOT NULL, 
	`ArtistId` BIGINT NOT NULL, 
	PRIMARY KEY (`AlbumId`), 
	CONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE artist (
	`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Name` VARCHAR(120), 
	PRIMARY KEY (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE customer (
	`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, 
	`FirstName` VARCHAR(40) NOT NULL, 
	`LastName` VARCHAR(20) NOT NULL, 
	`Company` VARCHAR(80), 
	`Address` VARCHAR(70), 
	`City` VARCHAR(40), 
	`State` VARCHAR(40), 
	`Country` VARCHAR(40), 
	`PostalCode` VARCHAR(10), 
	`Phone` VARCHAR(24), 
	`Fax` VARCHAR(24), 
	`Email` VARCHAR(60) NOT NULL, 
	`SupportRepId` BIGINT, 
	PRIMARY KEY (`CustomerId`), 
	CONSTRAINT customer_ibfk_1 FOREIGN KEY(`SupportRepId`) REFERENCES employee (`EmployeeId`)


In [22]:
def run_query(query):
    print(f"Query being executed:{query}")
    return db.run(query)

In [23]:
run_query("""
SELECT Artist.Name
FROM Artist
LIMIT 5;
""")

Query being executed:
SELECT Artist.Name
FROM Artist
LIMIT 5;



"[('AC/DC',), ('Accept',), ('Aerosmith',), ('Alanis Morissette',), ('Alice In Chains',)]"

### LLM Initilization

In [36]:
from langchain_groq import ChatGroq

GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [37]:
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def write_sql_query(llm):

    template="""
        Based on the table schema below , write a SQL Query
        that would answer the user's question.

        Schema:{schema}

        Question:{question}

        SQL Query:
    """

    prompt=ChatPromptTemplate.from_messages([
        ("system","""
            You are an expert SQL Query generator.
            Generate only valid MySQL SQL Queries.
            Do not include explainations or markdowns.
        """),
        ("human",template)
    ])

    return prompt|llm|StrOutputParser()

In [46]:
# Test generation

sql_chain=write_sql_query(llm)

response=sql_chain.invoke({
    "schema":get_schema(),
    "question":"Show all tracks by AC/DC"
})

print(response)

SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';


In [47]:
result=run_query(response)

print(result)

Query being executed:SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';
[(1, 'For Those About To Rock (We Salute You)', 'For Those About To Rock We Salute You'), (6, 'Put The Finger On You', 'For Those About To Rock We Salute You'), (7, "Let's Get It Up", 'For Those About To Rock We Salute You'), (8, 'Inject The Venom', 'For Those About To Rock We Salute You'), (9, 'Snowballed', 'For Those About To Rock We Salute You'), (10, 'Evil Walks', 'For Those About To Rock We Salute You'), (11, 'C.O.D.', 'For Those About To Rock We Salute You'), (12, 'Breaking The Rules', 'For Those About To Rock We Salute You'), (13, 'Night Of The Long Knives', 'For Those About To Rock We Salute You'), (14, 'Spellbound', 'For Those About To Rock We Salute You'), (15, 'Go Down', 'Let There Be Rock'), (16, 'Dog Eat Dog', 'Let There Be Rock'), (17, 'Let There Be Rock', 'Let There Be Rock